# Topic Modeling

Bu çalışmada bize örnek olarak verilen dokümanlarda geçen ifadelere topiclere ayırmaya çalışacağız. Ve bu işlemi yaparken de `LSA` ve `NMF` adını verdiğimiz temel 2 tane teknik öğreneceğiz. Hazırsanız başlayalım :)

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import Normalizer
from sklearn import metrics

## 1. Latent Semantic Analysis (LSA)

![LSA](https://res.cloudinary.com/dyd911kmh/image/upload/f_auto,q_auto:best/v1538411402/image3_maagmh.png)

İlk örnekte **Latent Semantic Analysis (LSA)** yöntemini kullanarak dokümanlarımız içerisinde topiclerimizi elde etmeye çalışacağız. Anlaşılırlığı olabildiğince yüksek tutmak için aşağıda görüldüğü gibi yalnızca 7 doküman içeren bir veri seti üzerinden çalışacağız.

In [2]:
example1 = ['Football baseball basketball',
            'baseball giants cubs redsox',
            'football broncos cowboys',
            'baseball redsox tigers',
            'pop stars hendrix prince',
            'hendrix prince jagger rock',
            'joplin pearl jam tupac rock']

In [3]:
vectorizer = CountVectorizer(stop_words='english')
doc_word = vectorizer.fit_transform(example1)
doc_word.shape

(7, 19)

Şu anda elimizde toplamda **7 doküman** ve bunun içerisinde yer alan **19 benzersiz (unique) kelime** bulunmakta. Yani aslında bizim buradaki temel amacımız bu **17 kelimeden istediğimiz sayıda topic** çıkarımında bulunmak!

In [4]:
columns1 = vectorizer.get_feature_names_out()
# CountVectorizer sonra elde ettiğimiz sparse matrisi istediğimiz çıktıyı vermediği için basit bir dönüşüm uyguluyoruz
pd.DataFrame(doc_word.toarray(), index=example1, columns=columns1)

,baseball,basketball,broncos,cowboys,cubs,football,giants,hendrix,jagger,jam,joplin,pearl,pop,prince,redsox,rock,stars,tigers,tupac
Football baseball basketball,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
baseball giants cubs redsox,1,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0
football broncos cowboys,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
baseball redsox tigers,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0
pop stars hendrix prince,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,1,0,0
hendrix prince jagger rock,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,1,0,0,0
joplin pearl jam tupac rock,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,1,0,0,1


In [5]:
lsa = TruncatedSVD(2) # 2 topic oluşturmak istiyorum
doc_topic = lsa.fit_transform(doc_word)
doc_topic

array([[ 7.09214927e-15,  1.07194739e+00],
       [ 1.24796176e-14,  1.73444730e+00],
       [ 1.99594844e-15,  3.31249959e-01],
       [ 1.03656944e-14,  1.40319735e+00],
       [ 1.42034317e+00, -1.05715861e-14],
       [ 1.69829182e+00, -1.25771077e-14],
       [ 1.22057878e+00, -8.68607046e-15]])

Bu çıktımızın okunmasını biraz daha kolaylaştıralım :)

In [6]:
lsa_df = pd.DataFrame(doc_topic.round(5),
             index = example1,
             columns = ["Topic 1","Topic 2"])
lsa_df

,Topic 1,Topic 2
Football baseball basketball,0.00000,1.07195
baseball giants cubs redsox,0.00000,1.73445
football broncos cowboys,0.00000,0.33125
baseball redsox tigers,0.00000,1.40320
pop stars hendrix prince,1.42034,-0.00000
hendrix prince jagger rock,1.69829,-0.00000
joplin pearl jam tupac rock,1.22058,-0.00000


İşte **LSA** yöntemi kullanarak oluşturduğumuz topiclere ilişkin tahminler! Görünüşe göre **ilk 4 doküman Topic 2**, **son 3 doküman Topic 1** olarak belirlenmiş! Girdi olarak verdiğimiz dokümanların içeriğine baktığımızda ise;

- Topic 1'in **müzik** konusuyla,
- Topic 2'nin ise **spor** konusuyla ilgili olduğunu görebiliyoruz.

Ayrıca yapılan tahminler de oldukça kendinden emin gözüküyor. Bu oldukça iyi birşey! Ancak unutmayın ki bu aşamada çok basit dokümanlarla çalışıyoruz. Gerçek verilerle çalıştığımızda bu kadar kararlı tahminler göremeyecek olmamız muhtemel :)

Şimdi de gelin, bu sonucun ortaya çıkmasında hangi kelime ne kadarlık bir rol üstlenmiş ona bakalım.

In [7]:
topic_word = pd.DataFrame(lsa.components_.round(3),
             index = ["Topic 1","Topic 2"],
             columns = columns1)
topic_word

,baseball,basketball,broncos,cowboys,cubs,football,giants,hendrix,jagger,jam,joplin,pearl,pop,prince,redsox,rock,stars,tigers,tupac
Topic 1,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.488,0.266,0.191,0.191,0.191,0.222,0.488,0.000,0.457,0.222,0.000,0.191
Topic 2,0.675,0.172,0.053,0.053,0.278,0.225,0.278,-0.000,-0.000,-0.000,-0.000,-0.000,-0.000,-0.000,0.503,-0.000,-0.000,0.225,-0.000


Sunumda konuştuğumuz **3 boyuttan 2 boyuta** düştüğümüz örneği hatırlıyor musunuz? Bize bir formül üretiyordu. Yani yukarıdaki tablomuz bize aslında şu çıktıyı veriyor:

$${Topic 1} = 0.000*(baseball) + 0.000*(basketball) +  ... + 0.488*(hendrix) + 0.266*(jagger) + ...$$

$${Topic 2} = 0.675*(baseball) + 0.172*(basketball) +  ... - 0.000*(hendrix) - 0.000*(jagger) + ...$$

Topiclerin oluşturulmasında kelimelerin üstlendiği role baktığımızda ise **Topic 1** için **hendrix** ve **prince** kelimelerinin ön plana çıktığını, **Topic 2** için ise özellikle **baseball** kelimesinin ağırlığının oldukça yüksek olduğunu görebiliyoruz!

Bu kısmı tamamlamadan önce daha çok kelimeyle çalıştığımız örneklerde, sonuçta en etkili olan kelimeleri daha kolay görebilmemizi sağlayan bir fonksiyon yazalım. Buradan çıktılarımızı daha net görebiliriz :)

In [8]:
def display_topics(model, feature_names, no_top_words, topic_names=None):
    for ix, topic in enumerate(model.components_):
        if not topic_names or not topic_names[ix]:
            print("Topic ", ix)
        else:
            print("Topic: '",topic_names[ix],"'")
        print(", ".join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))

In [9]:
display_topics(lsa, columns1, 3)

Topic  0
hendrix, prince, rock
Topic  1
baseball, redsox, cubs


## 2. Non-Negative Matrix Factorization (NMF)

![NMF](https://lh4.googleusercontent.com/FvUGnhGjYVFUY-K1pZxIrgPJKElEKxdnaX1GfDg3suLoMY5KUzgXKRsqcs-pDso-NXGyGdkq9r6S81G2Nc5zRaEQtQ88OfBSgMzRBk3RiZTXANxTYFzTLeWEoEZTOhxPH1UYCwqd)

İkinci örnekte ise **Non-Negative Matrix Factorization (NMF)** yöntemini kullanarak dokümanlarımız içerisinde topiclerimizi elde etmeye çalışacağız. Bu sefer de biraz daha uzun 6 tane doküman üzerinde çalışmalarımızı gerçekleştirelim.

In [10]:
example2 = ['Hamilton brought a boost. The Lion King provided ballast. And Broadway, once again, broke a record: The theater season that just ended attracted more people, and more money, than any before. Broadway seems to be defying the cultural odds: An ancient art form in the digital age, it is strengthening thanks to an everincreasing influx of tourists and a resurgent enthusiasm for musical theater. The season that ended on Sunday included 13,317,980 visitors to Broadway shows  a record number, up 1.6 percent over the previous season, according to figures released on Monday by the Broadway League. Theaters grossed 1.373 billion, also a record, up 0.6 percent over the previous season, although the grosses are not adjusted for inflation. Once again, Simba ruled supreme: The Lion King, still mighty more than 18 years after it opened, grossed 102.7 million on Broadway last season, far outpacing any other show. The musical, which has multiple productions running simultaneously around the globe, has grossed more than 6.2 billion worldwide, and has been seen by 85 million people over its history, according to Disney; by contrast, 478,605 people have seen the Broadway production of Hamilton thus far.Hamilton (featuring a onetime Simba, Christopher Jackson, in the role of George Washington) offered an enormous jolt of energy to the Broadway season. This hiphop musical about Americas founding fathers has dominated the cultural conversation, raked in awards and been celebrated at the White House. Many Broadway leaders believe the show has helped the industry as a whole, bringing attention from corners of the culture that have long preferred to mock jazz hands and dream ballets.',
          'When Candace Payne, aka the Chewbacca Mask Mom, sat in her car last Thursday filming her new Hasbro toy, an electronic Chewbacca Mask from Kohls, she inadvertently made history  not just for Facebook Live as its most popular video, but for the entire haul and unboxing video genre. Paynes video starts out like every other video in the genre  she talks about her shopping trip, and is incredibly excited to show the viewer her new purchase   but after that, the similarities stop. Shes not in a bedroom, but in her car, and Payne isnt describing multiple purchases, just one. The platform, execution and reception of her vlog has impacted the genre, in quite a few ways. First, Paynes video actually went viral among ordinary people, something that doesnt really happen to other haul and unboxing videos  not to this extent. While it is true boxing and haul videos by top YouTube vloggers will get a few million views (only!) thanks to the large communities said vlogger has built over the years, no one has ever seen an instant worldwide smash hit like Paynes video. Grandparents and aunts that dont even know what a haul video is were watching, liking, and sharing Paynes video.',
          'LOS ANGELES (AP)  An original animators desk from Walt Disney Studios and a vintage Mickey Mouse doll signed by Walt Disney are among the items up for bid next month in an online auction of rare Disney memorabilia. The website of Van Eaton Galleries lists more than 700 items for sale. Among the items listed are original production cels for Disney classics like The Jungle Book, Sleeping Beauty, Bambi and Snow White and the Seven Dwarfs. Collectors can also bid on costumes from the original Mickey Mouse Club, including one worn by Annette Funicello. An exhibition titled, Collecting Disney, opens Wednesday at the gallery in Sherman Oaks, California, ahead of the online auction that begins June 18.',
          'After putting together one of their best playoff performances in a must win Game 3 on Saturday, the Toronto Raptors picked up where they left off in Mondays Game 4, with AllStar guards Kyle Lowry and DeMar DeRozan finally teaming up for a complete performance. Lowry (35 points) and DeRozan (32 points) shot a combined 28 for 43 for 67 points and became the first teammates in a conference finals series to score 30  plus points on 60% or better shooting since Charles Barkley and Dan Majerle for Phoenix Suns in 1993, further proving that when the starting backcourt is on, the Raptors are extremely difficult to beat. Those numbers are of stark contrast to the majority of the Raptors first two playoff series, where both Lowry and DeRozan struggled mightily to deliver significant offensive production.',
          'The Cleveland Cavaliers enjoyed one of their most devastating 12 minutes of offensive basketball in the second half Monday night and, considering their playoff run, thats saying something. But it came after a long stretch of some of their most puzzling play in weeks, and that cost them a valuable playoff game. The Toronto Raptors evened the Eastern Conference finals at 22 with a 10599 victory after holding on in the face of a vicious Cavs late rally. Yet, as well as the Raptors played  stars Kyle Lowry and DeMar DeRozan were just terrific with a combined 67 points, the most theyve ever scored as teammates  it really came amid some headscratching, gameplan adjustments by coach Tyronn Lue. After spending the past few weeks finding a rhythm that has produced mostly spectacular results, Lue completely changed his rotations in the first half in what seemed like an overreaction from the Game 3 loss.',
          'Leave it to Rich Hill to end the As four game losing streak. The last time Oakland had won before Monday, Hill was on the mound. And at Safeco Field, he was magnificent, working calmly and efficiently whether the bases were empty or full. Hill pitched eight scoreless innings to help the As top the division leading Mariners 5 0. The As have won all four games theyve played at Seattle this season. Oakland has 20 wins and Hill has seven of them, the most for an As pitcher before the end of May since Mark Mulder had eight in 2003, a year Mulder made the All Star team. Every game he goes out there we feel were going to win, no matter what were going through, Oakland manager Bob Melvin said. He brings a lot of intensity to the mound, a lot of fight. Hill hasnt allowed more than three earned runs in any of his 10 starts and his ERA is down to 2.18. He also became the first As starter to pitch into the eighth inning since Sonny Gray pitched eight innings last Aug. 22, a span of 83 games; Melvin said his plan was to use only Hill and closer Ryan Madson, and Hill even wanted to go back out for the ninth after throwing 107 pitches. Hills streak of starts in which he gave up no more than four hits while working at least five innings ended at six; the Mariners recorded eight hits off him, few of them struck well. Hills streak was the best in franchise history dating to at least 1913. Seattle loaded the bases with no outs in the second inning without hitting the ball hard, with Nelson Cruzs infield single, an opposite field flare by Dae Ho Lee and a bloop to center by Kyle Seager. At that point, Hill said, second baseman Chris Coghlan came over to him and said, Control what you can control.']

doc_label = [doc[:25]+"..." for doc in example2] # Index değerlerinin oluşturulması

> **ÖNEMLİ NOT:** Bu çalışmada temel amacımız topic modelingde kullanılan yöntemleri kavramak olduğu için, önceki derslerimizde konuştuğumuz **ön işleme adımlarının bir çoğu bulunmamakta**! Çalışmalarınızı yaparken bu durumu göz önünde bulundurun :)

In [11]:
vectorizer = CountVectorizer(stop_words = 'english')
doc_word = vectorizer.fit_transform(example2)
columns2 = vectorizer.get_feature_names_out()
pd.DataFrame(doc_word.toarray(), index=doc_label, columns=columns2)

,10,102,10599,107,12,13,18,1913,1993,20,...,white,win,wins,won,working,worldwide,worn,year,years,youtube
Hamilton brought a boost....,0,1,0,0,0,1,1,0,0,0,...,1,0,0,0,0,1,0,0,1,0
"When Candace Payne, aka t...",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,1
LOS ANGELES (AP) An orig...,0,0,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,1,0,0,0
After putting together on...,0,0,0,0,0,0,0,0,1,0,...,0,1,0,0,0,0,0,0,0,0
The Cleveland Cavaliers e...,0,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Leave it to Rich Hill to ...,1,0,0,1,0,0,1,1,0,1,...,0,1,1,2,2,0,0,1,0,0


Bu sefer elimizdeki veri miktarı daha büyük! İlk örnekte 17 olan unique kelime sayımız, bu örnekte 468 olarak karşımıza çıkmış.

In [12]:
nmf_model = NMF(2) # 2 topic oluşturmak istiyorum
doc_topic = nmf_model.fit_transform(doc_word)

In [13]:
nmf_df = pd.DataFrame(doc_topic.round(5),
             index = doc_label,
             columns = ["Topic 1"," Topic 2"])
nmf_df

,Topic 1,Topic 2
Hamilton brought a boost....,3.32559,0.00000
"When Candace Payne, aka t...",0.52091,0.16900
LOS ANGELES (AP) An orig...,0.15392,0.02246
After putting together on...,0.02696,0.14048
The Cleveland Cavaliers e...,0.04443,0.20409
Leave it to Rich Hill to ...,0.00000,2.55847


Şimdi de **NMF** yöntemi kullanarak oluşturduğumuz topiclere ilişkin tahminleri görüyoruz. Bu sefer de görünüşe göre **ilk 3 doküman Topic 1**, **son 3 doküman ise Topic 2** olarak belirlenmiş!

Ama tabiki hangi kelimelerin bu seçimde nasıl bir rol oynadığını da görmemiz gerekiyor. Hemen onu inceyelim.

In [14]:
topic_word = pd.DataFrame(nmf_model.components_.round(3),
             index = ["Topic 1","Topic 2"],
             columns = columns2)
topic_word

,10,102,10599,107,12,13,18,1913,1993,20,...,white,win,wins,won,working,worldwide,worn,year,years,youtube
Topic 1,0.000,0.293,0.004,0.000,0.004,0.293,0.303,0.000,0.002,0.000,...,0.306,0.000,0.000,0.000,0.000,0.338,0.014,0.000,0.338,0.046
Topic 2,0.386,0.000,0.031,0.386,0.031,0.000,0.384,0.386,0.021,0.386,...,0.000,0.407,0.386,0.771,0.771,0.020,0.003,0.386,0.020,0.025


In [15]:
display_topics(nmf_model, columns2, 10)

Topic  0
broadway, season, people, record, grossed, hamilton, musical, million, seen, far
Topic  1
hill, said, oakland, innings, streak, game, second, starts, hits, mulder


Gördüğünüz gibi ilk örneğimizde elde ettiğimiz gibi çok net bir konu ayrımı yapmak pek mümkün gibi durmuyor. Bu da metin verileri üzerinde ön işleme işlemlerinin önemini tekrar gözlerimizin önüne seriyor :)

## 3. Metin Verileri İçerisinde Benzerlik

In [16]:
example3 = ["Machine learning is super fun",
"Python is super, super cool",
"Statistics is cool, too",
"Data science is fun",
"Python is great for machine learning",
"I like football",
"Football is great to watch"]

In [17]:
vectorizer = CountVectorizer(stop_words = 'english')
dtm = vectorizer.fit_transform(example3)
columns3 = vectorizer.get_feature_names_out()
pd.DataFrame(dtm.toarray(), index=example3, columns=columns3)

,cool,data,football,fun,great,learning,like,machine,python,science,statistics,super,watch
Machine learning is super fun,0,0,0,1,0,1,0,1,0,0,0,1,0
"Python is super, super cool",1,0,0,0,0,0,0,0,1,0,0,2,0
"Statistics is cool, too",1,0,0,0,0,0,0,0,0,0,1,0,0
Data science is fun,0,1,0,1,0,0,0,0,0,1,0,0,0
Python is great for machine learning,0,0,0,0,1,1,0,1,1,0,0,0,0
I like football,0,0,1,0,0,0,1,0,0,0,0,0,0
Football is great to watch,0,0,1,0,1,0,0,0,0,0,0,0,1


> **SORU:** Buradaki dokümanlardan 2 topic elde etmek istesem sizce bu topicler ne olmalı?

In [18]:
lsa = TruncatedSVD(2) # LSA yöntemi ile 2 topic
dtm_lsa = lsa.fit_transform(dtm)

In [19]:
pd.DataFrame(dtm_lsa, index=example3, columns=["Topic 1","Topic 2"])

,Topic 1,Topic 2
Machine learning is super fun,1.572473,-0.476507
"Python is super, super cool",2.015905,1.214919
"Statistics is cool, too",0.318174,0.466633
Data science is fun,0.294699,-0.297151
Python is great for machine learning,1.245758,-1.210149
I like football,0.037972,-0.381141
Football is great to watch,0.240585,-0.992333


In [20]:
pd.DataFrame(lsa.components_, index=["Topic 1","Topic 2"], columns=columns3)

,cool,data,football,fun,great,learning,like,machine,python,science,statistics,super,watch
Topic 1,0.280004,0.035353,0.033417,0.223993,0.178307,0.338085,0.004555,0.338085,0.391281,0.035353,0.038169,0.672310,0.028861
Topic 2,0.365270,-0.064548,-0.298349,-0.168056,-0.478428,-0.366379,-0.082792,-0.366379,0.001036,-0.064548,0.101363,0.424306,-0.215557


> **SORU:** Elde edilen sonuç hakkında ne düşünüyorsunuz? Sizce nasıl geliştirmeler yapılabilir?

Son olarak dokümanlarımız arasındaki benzerliklere de bakalım.

In [21]:
dtm_lsa = Normalizer(copy=False).fit_transform(dtm_lsa)
pd.DataFrame(cosine_similarity(dtm_lsa), index=example3, columns=example3)

,Machine learning is super fun,"Python is super, super cool","Statistics is cool, too",Data science is fun,Python is great for machine learning,I like football,Football is great to watch
Machine learning is super fun,1.000000,0.669981,0.299536,0.879823,0.888530,0.383455,0.507335
"Python is super, super cool",0.669981,1.000000,0.908975,0.236612,0.254682,-0.428723,-0.299838
"Statistics is cool, too",0.299536,0.908975,1.000000,-0.189940,-0.171606,-0.766296,-0.670217
Data science is fun,0.879823,0.236612,-0.189940,1.000000,0.999826,0.776342,0.855956
Python is great for machine learning,0.888530,0.254682,-0.171606,0.999826,1.000000,0.764458,0.846169
I like football,0.383455,-0.428723,-0.766296,0.776342,0.764458,1.000000,0.990417
Football is great to watch,0.507335,-0.299838,-0.670217,0.855956,0.846169,0.990417,1.000000


## 4. Latent Dirichlet Allocation (LDA)

Bu çalışmada ise Sklearn kütüphanesinin bizlere sağlamış olduğu veri setlerinden biri olan [20 Newsgroups Dataset](https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html) ile işlemlerimizi yapacağız.

In [22]:
from gensim import corpora, models, similarities, matutils
from sklearn import datasets

In [23]:
# Veri seti içerisinden kullanılacak başlık kategorileri
categories = ['alt.atheism', 'comp.graphics', 'rec.sport.baseball', 
              'rec.motorcycles', 'sci.space', 'talk.politics.mideast']

# Header, footers ve quotes gibi bilgilere ihtiyacımız yok
ng_train = datasets.fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))

Şimdi de biraz veri setinin içeriğini tanımaya çalışalım.

In [24]:
ng_train.data[:3]

['Well, the Red Sox have apparenly resigned Herm Winningham to a AAA contract.\nTed "Larry" Simmons signed him to a AAA contract then released him from\nBuffalo, allowing Lou "Curly" Gorman to circumvent the rule about not\nresigning free agents until May 1. Clearly, neither of these guys is bright\nenough to be Moe.\n\n Mike Jones | AIX High-End Development | mjones@donald.aix.kingston.ibm.com',
 "I was wondering if anyone knows where I can get more information about\nthe graphics in the WingCommander series, and the RealSpace system they use.\nI think it's really awesome, and wouldn't mind being able to use similar\nfeatures in programs.  Thanks in advance.\n",
 '\n: 3DO is still a concept.\n: The software is what sells and what will determine its\n: success.\n\n\nApparantly you dont keep up on the news.  3DO was shown\nat CES to developers and others at private showings.  Over\n300 software licensees currently developing software for it.']

In [25]:
vectorizer = CountVectorizer(ngram_range=(1,2), stop_words='english', token_pattern="\\b[a-z][a-z]+\\b")
# token_pattern="\\b[a-z][a-z]+\\b" = içerisinde en az 2 karakter geçen kelimeleri al
doc_word = vectorizer.fit_transform(ng_train.data)
columns4 = vectorizer.get_feature_names_out()
pd.DataFrame(doc_word.toarray(), columns=columns4)

,aa,aa aaa,aa albany,aa atlanta,aa does,aa limited,aa moved,aa nd,aa play,aa season,...,zx pilot,zx spend,zx tried,zx ve,zygot,zygot ati,zyxel,zyxel epimntl,zyxel mnp,zzzzzzt
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3411,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3412,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3413,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3414,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Bu sefer ise **272.502** unique n-gramlardan bahsediyoruz, bu sefer işimiz biraz daha zor! Ama gelin **LDA** yönteminin nasıl bir çözüm izlediğine bakalım.

In [26]:
from gensim import corpora, models, similarities, matutils

corpus = matutils.Sparse2Corpus(doc_word) # Doc-Word matrisinin gensime çevrilmesi

# Kelime idlerini görmek yerine direk kelimelere görmek için dictionary
id2word = dict((v, k) for k, v in vectorizer.vocabulary_.items())

id2word

{197236: 'red',
 224689: 'sox',
 9685: 'apparenly',
 201872: 'resigned',
 106244: 'herm',
 266110: 'winningham',
 11: 'aaa',
 48819: 'contract',
 239388: 'ted',
 131411: 'larry',
 219684: 'simmons',
 219236: 'signed',
 198940: 'released',
 30183: 'buffalo',
 6699: 'allowing',
 141674: 'lou',
 54172: 'curly',
 97545: 'gorman',
 39435: 'circumvent',
 207021: 'rule',
 201876: 'resigning',
 90324: 'free',
 4691: 'agents',
 40718: 'clearly',
 101536: 'guys',
 29223: 'bright',
 153780: 'moe',
 150980: 'mike',
 124193: 'jones',
 5726: 'aix',
 106697: 'high',
 73987: 'end',
 61770: 'development',
 153102: 'mjones',
 68128: 'donald',
 128175: 'kingston',
 111092: 'ibm',
 42912: 'com',
 197273: 'red sox',
 224690: 'sox apparenly',
 9686: 'apparenly resigned',
 201874: 'resigned herm',
 106245: 'herm winningham',
 266111: 'winningham aaa',
 16: 'aaa contract',
 48841: 'contract ted',
 239397: 'ted larry',
 131421: 'larry simmons',
 219695: 'simmons signed',
 219237: 'signed aaa',
 48835: 'contrac

In [27]:
lda = models.LdaModel(corpus=corpus, num_topics=3, id2word=id2word) # 3 topic oluşturmak istiyorum

In [28]:
lda.print_topics(num_words=10) # En önemli 10 kelimeyi getir

[(0,
  '0.033*"accusations make" + 0.027*"address request" + 0.026*"added jesus" + 0.015*"acdis campus" + 0.014*"ac bus" + 0.013*"act certain" + 0.012*"abducted" + 0.012*"adequate evidence" + 0.011*"absolutely merely" + 0.011*"able star"'),
 (1,
  '0.038*"active public" + 0.018*"actually stance" + 0.012*"ability actively" + 0.012*"adjacent row" + 0.010*"account women" + 0.009*"accelerated" + 0.009*"ad pc" + 0.008*"absood" + 0.008*"address promised" + 0.007*"abridgment"'),
 (2,
  '0.039*"achim stoesser" + 0.032*"adams said" + 0.026*"access paper" + 0.023*"access baud" + 0.016*"accor higher" + 0.014*"account persecution" + 0.013*"activities grant" + 0.013*"activities reveal" + 0.009*"according religion" + 0.008*"acceptably"')]

Artık **LDA** modelimiz ve buna ilişkin istediğimiz 3 tane topic oluşmuş durumda. O zaman hadi bunlara ilişkin tahminleri görelim.

In [29]:
lda_corpus = lda[corpus]
lda_docs = [doc for doc in lda_corpus]

In [30]:
lda_docs[0:3]
# İlk Doküman ==> %62-->0. Topic / %10-->1. Topic / %28--> 2. Topic
# İkinci Doküman ==> %44-->0. Topic / %11-->1. Topic / %45--> 2. Topic
# Üçüncü Doküman ==> %17-->0. Topic / %17-->1. Topic / %66--> 2. Topic

[[(0, 0.35998207), (1, 0.6122123), (2, 0.027805647)],
 [(0, 0.44348082), (1, 0.44534737), (2, 0.11117175)],
 [(0, 0.16675161), (1, 0.6664794), (2, 0.16676901)]]

In [31]:
ng_train.data[0:3]

['Well, the Red Sox have apparenly resigned Herm Winningham to a AAA contract.\nTed "Larry" Simmons signed him to a AAA contract then released him from\nBuffalo, allowing Lou "Curly" Gorman to circumvent the rule about not\nresigning free agents until May 1. Clearly, neither of these guys is bright\nenough to be Moe.\n\n Mike Jones | AIX High-End Development | mjones@donald.aix.kingston.ibm.com',
 "I was wondering if anyone knows where I can get more information about\nthe graphics in the WingCommander series, and the RealSpace system they use.\nI think it's really awesome, and wouldn't mind being able to use similar\nfeatures in programs.  Thanks in advance.\n",
 '\n: 3DO is still a concept.\n: The software is what sells and what will determine its\n: success.\n\n\nApparantly you dont keep up on the news.  3DO was shown\nat CES to developers and others at private showings.  Over\n300 software licensees currently developing software for it.']

## Topic Modeling Özet

1. Metin verileri üzerinde gerekli olan **ön işleme adımlarını** gerçekleştirin
2. Kullanacağınız algoritmayı seçin (**LSA, NMF, LDA**)
3. Modellerin oluşturulabilmesi için gerekli olan **dönüşümleri** uygulayın
4. Anlamlı topicler elde edilene kadar parametreleri değiştirerek **en iyi sonuca** ulaşmaya çalışın
5. Elde edilen topicler içinize sindiğinde işlem tamam :)


## Algoritma Seçimi

- **LSA** ve **NMF** algoritmaları daha **küçük çarplı** çalışmalarda daha uygundurlar (Tweetler, Yorumlar)
- **LDA** ise **büyük dokümanlarla** çalışırken en çok tercih edilen yöntemdir (Kitaplar)
- Ama biz çalışmalarımızda, bu öğrendiğimiz **3 tekniği de uygulayarak** hangisinden daha iyi sonuç aldığımıza odaklanmamız gerekiyor

![LSA_NMF_LDA](https://www.researchgate.net/publication/333865994/figure/fig3/AS:771340751630337@1560913374509/Topic-Modeling-using-LSA-NMF-and-LDA-After-topic-modeling-we-identify-topic-topics.ppm)